### RAG Pipeline from Data Ingestion to VectorDB


In [6]:
import os 
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


## Read all the pdf in the directory

In [7]:
def process_all_pdf(pdf_directory):
    """Process all pdf files in the given directory and return a list of documents."""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    """Finds all pdf files recursively """
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing : {pdf_file.name}")
        try:
            loader =PyPDFLoader(str(pdf_file))
            documents = loader.load()

            """Add source information to the metadata"""
            for doc in documents:
                doc.metadata["source"]=pdf_file.name
                doc.metadata["path"] = "pdf"

            all_documents.extend(documents)
            print(f"Processed {len(documents)} pages")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"Total documents processed: {len(all_documents)}")
    return all_documents
"""Process all PDFs in the data directory """
all_pdf_documents = process_all_pdf("../data")


Found 2 PDF files to process.

Processing : Final Year Y.B.Tech__CSE_Structure & Syllabus_NEP.pdf
Processed 79 pages

Processing : SEM 6 Syllabus (1).pdf
Processed 9 pages
Total documents processed: 88


In [8]:
print(len(all_pdf_documents))

88


## Text Chunking using Text Splitter

In [9]:
""" Text Splitting into chunks"""
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks to get the better performance by RAG Model"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]   
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Total {len(documents)} documents are split into {len(split_docs)} chunks.")

    if split_docs:
        print("\nExample chunk:")
        print(f"Content:{split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

chunks = split_documents(all_pdf_documents)
chunks


Total 88 documents are split into 169 chunks.

Example chunk:
Content:Government College of Engineering Aurangabad, 
ChhatrapatiSambhajinagar 
 
(An Autonomous Institute of the Government of Maharashtra) 
Station Road, Osmanpura, Aurangabad – 431005 (M.S.) 
Phone – (024...
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-01T14:59:34+05:30', 'moddate': '2026-03-01T15:01:23+05:30', 'author': 'Admin', 'source': 'Final Year Y.B.Tech__CSE_Structure & Syllabus_NEP.pdf', 'total_pages': 79, 'page': 0, 'page_label': '1', 'path': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-01T14:59:34+05:30', 'moddate': '2026-03-01T15:01:23+05:30', 'author': 'Admin', 'source': 'Final Year Y.B.Tech__CSE_Structure & Syllabus_NEP.pdf', 'total_pages': 79, 'page': 0, 'page_label': '1', 'path': 'pdf'}, page_content='Government College of Engineering Aurangabad, \nChhatrapatiSambhajinagar \n \n(An Autonomous Institute of the Government of Maharashtra) \nStation Road, Osmanpura, Aurangabad – 431005 (M.S.) \nPhone – (0240) 2366101, 2366111, Fax (0240) 2332835  \n \nDepartment of Computer Science and Engineering \nCurriculum  \nFinal Year, B. Tech. (CSE) \n \nFrom Academic Year 2026-27  \nAs per NEP'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-01T14:59:34+05:30', 'moddate': '2026-03-01T15:01:23+05:30', 'author': 'Admin', 'source': 'Final Year 